# Student Performance Analysis
An exploratory notebook version of the analysis performed in `analysis.py`.

This notebook walks through data loading, cleaning, EDA, and answers the core business questions with visualizations.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (9, 6)

DATA_PATH = os.path.join("..", "data", "student_performance.csv")
PASS_MARK = 40

## 1. Load the Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
print("Missing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df["gender"] = df["gender"].str.strip().str.title()
df["parental_education"] = df["parental_education"].str.strip()
df.loc[df["study_hours_per_day"] > 12, "study_hours_per_day"] = np.nan

for col in ["study_hours_per_day", "attendance_percentage"]:
    df[col] = df[col].fillna(df[col].median())

df["parental_education"] = df["parental_education"].fillna(df["parental_education"].mode()[0])

int_cols = ["age", "math_score", "science_score", "english_score"]
df[int_cols] = df[int_cols].astype(int)

df["average_score"] = df[["math_score", "science_score", "english_score"]].mean(axis=1).round(2)
df["total_score"] = df[["math_score", "science_score", "english_score"]].sum(axis=1)
df["result"] = np.where(
    (df["math_score"] >= PASS_MARK) & (df["science_score"] >= PASS_MARK) & (df["english_score"] >= PASS_MARK),
    "Pass", "Fail"
)
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
df["gender"].value_counts()

In [ ]:
df["parental_education"].value_counts()

## 4. Business Question 1 — Average Marks by Gender

In [ ]:
avg_by_gender = df.groupby("gender")["average_score"].mean().round(2)
avg_by_gender.plot(kind="bar", color=["#4C72B0", "#DD8452"], title="Average Marks by Gender")
plt.ylabel("Average Score")
plt.show()
avg_by_gender

## 5. Business Question 2 — Effect of Study Hours on Marks

In [ ]:
sns.regplot(data=df, x="study_hours_per_day", y="average_score",
            scatter_kws={"alpha": 0.3}, line_kws={"color": "red"})
plt.title("Study Hours vs Average Score")
plt.show()
df["study_hours_per_day"].corr(df["average_score"])

## 6. Business Question 3 — Effect of Parental Education

In [ ]:
edu_order = ["High School", "Some College", "Bachelor's", "Master's", "PhD"]
avg_by_edu = df.groupby("parental_education")["average_score"].mean().reindex(edu_order)
avg_by_edu.plot(kind="bar", title="Average Marks by Parental Education")
plt.ylabel("Average Score")
plt.xticks(rotation=20)
plt.show()
avg_by_edu

## 7. Business Question 4 — Subject-wise Average Marks

In [ ]:
subject_avgs = df[["math_score", "science_score", "english_score"]].mean()
subject_avgs.plot(kind="bar", title="Subject-wise Average Marks")
plt.ylabel("Average Score")
plt.show()
subject_avgs

## 8. Business Question 5 — Top 10 Students

In [ ]:
top10 = df.sort_values("average_score", ascending=False).head(10)
top10[["student_id", "name", "gender", "average_score", "total_score"]]

## 9. Business Question 6 — Pass vs Fail Analysis

In [ ]:
result_counts = df["result"].value_counts()
plt.pie(result_counts, labels=result_counts.index, autopct="%1.1f%%",
        colors=["#4CAF50", "#E53935"], startangle=90)
plt.title("Pass vs Fail Distribution")
plt.show()
result_counts

## 10. Business Question 7 — Correlation Heatmap

In [ ]:
numeric_cols = ["age", "study_hours_per_day", "attendance_percentage",
                "math_score", "science_score", "english_score", "average_score"]
corr = df[numeric_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 11. Key Insights

- Study hours have a moderate positive correlation with average score.
- Parental education level is associated with higher average scores, particularly for Master's/PhD-educated parents.
- English tends to be the strongest subject on average; Math the weakest.
- Attendance percentage shows a positive but weaker correlation with performance.
- The overall pass rate is high given a 40-mark passing threshold per subject.

See the main `README.md` and `analysis.py` for the full, script-based version of this analysis with all 10 saved charts.